# M3 response retrieval (Q1): R0 lookup, R1 BM25, R2 int8 sentence embeddings

After the verdict the app shows passages of the WHO South-East Asia snakebite guideline (2nd edition, 2016) word
for word, with their page, and one species card per class. Nothing is generated. Q1 asks which retriever finds the
right passages for a question within the 30 MB package: the fixed R0 lookup, R1 BM25, or R2 int8 sentence embeddings
from `all-MiniLM-L6-v2` (R2-L6) or `paraphrase-MiniLM-L3-v2` (R2-L3).

**Inputs:** the guideline PDF at `refs/who_searo_2016.pdf` beside the repo (or wherever `SERPENTGUARD_WHO_PDF`
points) and pdftotext; the passage index `data/m3/kb/who_passages.csv`, which holds ids, pages, character offsets,
our own headings and a hash per passage, never the text; the species cards in `data/m3/kb/species_cards.csv`, in our
own words from repo data; and the 60 draft questions in `data/m3/response_questions.csv`. There is no Kaggle input.

**Outputs:** `results/m3/q1/retrieval.csv` (Recall@1, Recall@3, MRR over the top 10, ms per question on one CPU
thread and MB on the device), `ranks.csv` (each question's gold rank), `paired.csv` (sign tests against R1) and
`versions.json` (packages, pdftotext, and hashes of the PDF, index, cards, questions and int8 models). The int8
models go to `kaggle/m3_runs/q1/` beside the repo and are never committed.

**Run:** locally on the CPU, from the `notebooks/` folder, with no parameters. The first run exports the two R2
models; a rerun takes under a minute. No cell prints guideline text: only ids, pages, counts and metrics.

In [1]:
import json
import sys
from pathlib import Path

SRC = Path("../src")
if SRC.is_dir() and str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

import pandas as pd

from serpentguard.respond import CARD_SOURCE, CARDS_CSV, INDEX_CSV, WHO_SOURCE, build_kb, find_who_pdf, read_index

QUESTIONS_CSV = Path("../data/m3/response_questions.csv")
RESULTS_DIR = Path("../results/m3/q1")     # committed: retrieval.csv, ranks.csv, paired.csv and versions.json
Q1_DIR = Path("../../kaggle/m3_runs/q1")   # beside the repo, never committed: the int8 models under models/<name>
REBUILD_KB = False                         # True writes the rebuilt index and cards over the committed CSVs

PDF = find_who_pdf()
assert PDF is not None, "put the guideline at refs/who_searo_2016.pdf beside the repo or set SERPENTGUARD_WHO_PDF"
print("guideline PDF:", PDF.name, "| int8 models under", Q1_DIR.name, "| results under results/m3/q1")

guideline PDF: who_searo_2016.pdf | int8 models under q1 | results under results/m3/q1


## 1. Knowledge base

The app reads two committed files: the passage index and the species cards. Both are built here.

**The passage index.** We cut the guideline's pages into passages from `pdftotext -raw` output: the pages in
`KB_PAGES`, in the sections of `SECTIONS` (our own heading and topic for each), with running heads, page numbers and
side-tab titles left out. Lines join into paragraphs, bullets and figure captions; a caption that interrupts a sentence
is folded into it, short list items join into one list, a short block joins the next one in its section, and a passage
over 900 characters is cut after its last sentence that fits. A passage that calls a snake harmless is left out. The
index keeps each passage's page, character offsets, heading, topic and a hash, never the text.

**The species cards**, one per class in our own words: the common name from the M1 and M2 class tables, the family,
the WHO category in India, whether the danger table calls it dangerous and why, and the M1 lookalike table's features
that tell it apart from its lookalikes.

At run time `build_kb` cuts every indexed passage from `pdftotext -raw` output of its page and checks it against the
stored hash, so another printing of the PDF or another pdftotext fails loudly instead of quoting the wrong words.

In [2]:
import functools
import re
from collections import Counter
from dataclasses import dataclass, replace

from serpentguard.medical_importance import PRINTED_PAGE_OFFSET
from serpentguard.respond import page_text, text_hash

INDEX_COLUMNS = ("id", "page", "printed_page", "start", "end", "section", "heading", "topic", "kind", "sha256")

# PDF page -> the section it opens in (None: carried on from the previous page); numbered headings move it on
KB_PAGES = {22: "2", 23: None, 24: None, 25: None,
            30: "3", 31: None, 33: "3.1", 34: None,
            36: "3.2.1", 37: None, 38: None, 41: None, 42: None, 44: None, 45: None, 52: None, 53: None, 54: None,
            55: None, 57: None, 58: None, 65: "3.2.1", 66: None, 67: None, 68: None, 69: None, 70: None,
            97: "5", 98: None, 99: None, 100: None, 102: None, 103: None, 104: None, 105: None, 106: None,
            107: "5.5",
            110: "6", 118: "6", 119: None, 120: None, 166: "6.11", 167: None,
            198: "A4", 199: None}
SECTIONS = {  # section -> (our heading, topic); passages in any other section are left out
    "2": ("prevention essentials", "prevention"), "2.1": ("lowering the risk of bites", "prevention"),
    "2.2": ("community education", "prevention"),
    "3": ("venomous snake essentials", "snakes"), "3.1": ("fangs and venom glands", "snakes"),
    "3.2": ("venomous snake groups", "snakes"), "3.2.1": ("species of concern", "snakes"),
    "3.2.2": ("other dangerous species", "snakes"), "3.2.3": ("snakes without venom", "snakes"),
    "3.3": ("telling snakes apart", "identification"),
    "5": ("clinical effects", "symptoms"), "5.1": ("no venom injected", "symptoms"),
    "5.2": ("venom injected", "symptoms"), "5.2.1": ("early signs", "symptoms"),
    "5.2.2": ("patterns of envenoming", "symptoms"), "5.2.3": ("signs at the bite", "symptoms"),
    "5.2.4": ("signs across the body", "symptoms"), "5.3": ("symptom patterns by snake", "symptoms"),
    "5.4": ("lasting harm", "aftercare"),
    "5.6": ("venom in the eyes", "eye"),
    "6": ("management essentials", "first_aid"), "6.1": ("stages of care", "first_aid"),
    "6.2": ("first aid", "first_aid"), "6.2.1": ("first aid principles", "first_aid"),
    "6.2.2": ("breathing and shock", "first_aid"), "6.2.3": ("steps to take", "first_aid"),
    "6.3": ("getting to hospital", "transport"),
    "6.12": ("rehabilitation", "aftercare"), "6.13": ("going home", "aftercare"),
    "6.14": ("cobra spit in the eyes", "eye"),
    "A4": ("pressure pad and bandage", "first_aid"),
}
LEFT_OUT = re.compile(r"\bharmless\b", re.IGNORECASE)  # a passage calling some snakes harmless never reaches a user

HEADING = re.compile(r"^(\d+(?:\.\d+)+)\.?\s+[A-Z]")
BULLET = re.compile(r"^[•▪●]")
FIGURE = re.compile(r"^(?:Figures?|Table)\s+\d")
LINE_END = re.compile(r"[.!?:]['’”)\]]*$")
SENTENCE_END = re.compile(r"[.!?]['’”)]*\s+(?=[A-Z“‘(•])")
PAGE_NUMBER = re.compile(r"^\s*\d+\s*\x0c?$")
PANEL_LABELS = re.compile(r"^\s*(?:\[[a-z]\]\s*)+$")
GLUED_MIN, GLUED_PART_MIN, GLUED_WORD_MIN = 15, 2, 3  # side-tab lines glue a title's words into one long token
GLUED_JOINERS = frozenset({"of", "and", "the", "in"})
EDGE_NUMBER = re.compile(r"^\d+\s*|\s*\d+$")
EDGE_LINES, HEAD_REPEAT, HEAD_MIN_CHARS = 3, 3, 5  # running heads: repeated at a page's edge on >= 3 pages
COLUMN_QUANTILE, WIDE_LINE = 0.95, 1.5   # column width: this quantile of lines up to 1.5x the median length
MIN_CHARS, MAX_CHARS, SHORT_ITEM, MIN_KEEP = 200, 900, 120, 60
MIN_LETTER_SHARE = 0.6


def index_passages(pdf):
    """The passage index: one row per passage, no text."""
    heads, words, rows, section = running_heads(pdf), document_words(pdf), [], None
    for page, opens_in in KB_PAGES.items():
        section = opens_in or section
        text = page_text(pdf, page)
        spans, section = _page_spans(text, heads, words, section)
        kept = [(start, end, sec, kind) for start, end, sec, kind in spans if _keep(text[start:end], sec)]
        for number, (start, end, sec, kind) in enumerate(kept, 1):
            heading, topic = SECTIONS[sec]
            rows.append({"id": f"who-{page:03d}-{number:02d}", "page": page, "printed_page": page - PRINTED_PAGE_OFFSET,
                         "start": start, "end": end, "section": sec, "heading": heading, "topic": topic, "kind": kind,
                         "sha256": text_hash(text[start:end])})
    return pd.DataFrame(rows, columns=INDEX_COLUMNS)


def running_heads(pdf):
    """Lines (page numbers stripped) found among the first or last lines of at least HEAD_REPEAT pages."""
    counts = Counter()
    for page in page_text(pdf).split("\x0c"):
        lines = [line for line in page.split("\n") if line.strip()]
        edges = {EDGE_NUMBER.sub("", line).strip() for line in lines[:EDGE_LINES] + lines[-EDGE_LINES:]}
        counts.update(line for line in edges if len(line) >= HEAD_MIN_CHARS and not line.startswith("("))
    return frozenset(line for line, count in counts.items() if count >= HEAD_REPEAT)


@functools.cache
def document_words(pdf):
    return Counter(re.findall(r"[a-z]+", page_text(pdf).lower()))


@dataclass
class _Block:
    kind: str
    start: int
    end: int
    first: str
    last: str
    section: str
    fresh: bool    # follows a non-body line, so it never merges into the block before

    @property
    def size(self):
        return self.end - self.start


def _page_spans(text, heads, words, section):
    """(start, end, section, kind) per passage on one page, and the section the page ends in."""
    lines, position = [], 0
    for line in text.split("\n"):
        stripped_line = line.rstrip("\x0c")
        lines.append((position, position + len(stripped_line), stripped_line, _is_body(stripped_line, heads, words)))
        position += len(line) + 1
    lengths = sorted(len(line) for _, _, line, body in lines if body)
    if not lengths:
        return [], section
    body_lengths = [n for n in lengths if n <= WIDE_LINE * lengths[len(lengths) // 2]]
    width = body_lengths[int(COLUMN_QUANTILE * (len(body_lengths) - 1))]
    blocks, current, kind, fresh = [], [], None, True

    def close():
        nonlocal fresh
        if current:
            blocks.append(_Block(kind, current[0][0], current[-1][1], current[0][2], current[-1][2], section, fresh))
            current.clear()
            fresh = False

    for i, (start, end, line, body) in enumerate(lines):
        if not body:
            close()
            fresh = True
            continue
        stripped = line.strip()
        after = next((other.strip() for _, _, other, ok in lines[i + 1:] if ok), "")
        heading = HEADING.match(stripped)
        if heading:
            close()
            kind, section = "heading", heading.group(1)
        elif BULLET.match(stripped):
            close()
            kind = "bullet"
        elif FIGURE.match(stripped):
            close()
            kind = "figure"
        elif not current:
            kind = "paragraph"
        current.append((start, end, line))
        if kind == "figure":
            if stripped.endswith("]") or (stripped.endswith(")") and "Copyright" in stripped and not after.startswith("[")):
                close()
        elif LINE_END.search(stripped) and (len(stripped) + 1 + len(after.split()[0] if after else "") <= width
                                            or not after[:1].isalnum()):
            close()
    close()
    spans = []
    for block in _merge(_fold_figures(blocks)):
        spans += [(a, b, block.section, block.kind) for a, b in _split_long(text, block.start, block.end)]
    return spans, section


def _is_body(line, heads, words):
    stripped = line.strip()
    return bool(stripped) and not (PAGE_NUMBER.match(line) or PANEL_LABELS.match(line)
                                   or EDGE_NUMBER.sub("", stripped).strip() in heads
                                   or any(_glued(token.lower(), words) for token in re.findall(r"[A-Za-z]+", stripped)))


def _glued(token, words):
    """True when token splits into guideline words, a joining word among them after the first (side-tab titles)."""
    if len(token) < GLUED_MIN:
        return False
    reached = [set() for _ in range(len(token) + 1)]  # at each cut: whether a joining word has been used
    reached[0].add(False)
    for end in range(1, len(token) + 1):
        for start in range(end):
            part, joiner = token[start:end], token[start:end] in GLUED_JOINERS and start > 0
            if reached[start] and words[part] >= GLUED_PART_MIN and (joiner or len(part) >= GLUED_WORD_MIN):
                reached[end] |= {joined or joiner for joined in reached[start]}
    return True in reached[-1]


def _fold_figures(blocks):
    """A paragraph that a figure caption interrupts mid-sentence becomes one passage, caption included."""
    folded = []
    for block in blocks:
        if block.kind == "paragraph" and block.first.strip()[:1].islower():
            before = len(folded) - 1
            while before >= 0 and folded[before].kind == "figure":
                before -= 1
            if before >= 0 and not LINE_END.search(folded[before].last.strip()):
                folded[before] = replace(folded[before], end=block.end, last=block.last)
                del folded[before + 1:]
                continue
        folded.append(block)
    return folded


def _merge(blocks):
    """Short list items join into one list; a short block joins the one after it in the same section."""
    merged = []
    for block in blocks:
        prev = merged[-1] if merged and not block.fresh else None
        if prev is not None:
            same = prev.section == block.section
            if (same and block.kind == "bullet" and prev.kind in ("bullet", "list") and block.size < SHORT_ITEM
                    and block.end - prev.start <= MAX_CHARS):
                merged[-1] = replace(prev, kind="list", end=block.end, last=block.last)
                continue
            if (prev.size < MIN_CHARS and "figure" not in (block.kind, prev.kind) and block.end - prev.start <= MAX_CHARS
                    and (same or prev.kind == "heading") and not (block.kind == "heading" and prev.kind != "heading")):
                merged[-1] = replace(prev, kind=block.kind if prev.kind == "heading" else prev.kind, end=block.end,
                                     last=block.last, section=block.section)
                continue
        merged.append(block)
    return merged


def _split_long(text, start, end):
    """Spans of at most MAX_CHARS, cut after the last sentence end that fits, else after the last line that fits."""
    spans = []
    while end - start > MAX_CHARS:
        cuts = [m.end() for m in SENTENCE_END.finditer(text, start, end) if MIN_CHARS <= m.end() - start <= MAX_CHARS]
        cuts = cuts or [i + 1 for i in range(start + MIN_CHARS, start + MAX_CHARS) if text[i] == "\n"]
        if not cuts:
            break
        spans.append((start, start + len(text[start:cuts[-1]].rstrip())))
        start = end - len(text[cuts[-1]:end].lstrip())
    return spans + [(start, start + len(text[start:end].rstrip()))]


def _keep(passage, section):
    letters = sum(char.isalpha() for char in passage)
    visible = sum(not char.isspace() for char in passage)
    return (section in SECTIONS and len(passage) >= MIN_KEEP and letters >= MIN_LETTER_SHARE * visible
            and not LEFT_OUT.search(passage))

In [3]:
import re
from pathlib import Path

import pandas as pd

from serpentguard.decision import BASIS_NOTE, NOT_LISTED_TEXT
from serpentguard.medical_importance import CLASSES_CSV, GUIDELINE, MEDICAL_CSV
from serpentguard.respond import REPO_ROOT

M1_REPORT = REPO_ROOT / "documentation" / "milestone 1" / "project-report.md"
M2_REPORT = REPO_ROOT / "documentation" / "milestone 2" / "project-report.md"
CARD_COLUMNS = ("id", "class_id", "species", "common_name", "family", "who_category", "dangerous", "basis", "text")
M1_CLASS_ROW = re.compile(r"^\|\s*\d+\s*\|\s*\*([^*]+)\*\s*\|\s*([^|]+?)\s*\|")
M2_CLASS_ROW = re.compile(r"^\|\s*\*([^*]+)\*,\s*([^|]+?)\s*\|")
LOOKALIKE_SECTION, LOOKALIKE_HEADER = "### 3.6 Lookalike species pairs", "| Pair"
GUIDELINE_CITATION, REPORT_CITATION = re.compile(r"\[15, (PDF pp?\. [\d, ]+)\]"), re.compile(r"\[(\d+)\]")
LOOKALIKE_ROWS = (  # the species in each row of the M1 lookalike table, in its order
    ("Naja naja", "Ptyas mucosa"),
    ("Bungarus caeruleus", "Lycodon aulicus", "Lycodon travancoricus", "Lycodon fasciolatus"),
    ("Daboia russelii", "Eryx conicus", "Python molurus"),
    ("Echis carinatus", "Boiga trigonata"),
    ("Naja naja", "Fowlea piscator"),
    ("Craspedocephalus gramineus", "Rhabdophis plumbicolor"),
    ("Hypnale hypnale", "Craspedocephalus malabaricus", "Echis carinatus"),
)
LOOKALIKE_REWORDING = {  # row -> (M1 words, ours): M1's phrasing there runs five words with the guideline's
    0: (("scale between the nasal", "scale separating the nasal"),),
    3: (("rough scales on top", "rough scales covering the crown"),),
    5: (("nostril and the eye", "eye and the nostril"),),
}
CATEGORY_TEXT = {"1": "Category 1", "2": "Category 2", "not_listed": "not listed"}


def common_names(sources=(M1_REPORT, M2_REPORT)):
    """Common names from the M1 class table first, then M2's WHO-listed table."""
    names = {}
    for path in sources:
        for line in Path(path).read_text().splitlines():
            match = M1_CLASS_ROW.match(line) or M2_CLASS_ROW.match(line)
            if match:
                names.setdefault(match.group(1).strip(), match.group(2).strip())
    return names


def lookalike_features(report=M1_REPORT):
    """species -> [(other species, what tells them apart)] from the M1 lookalike table."""
    lines = Path(report).read_text().split(LOOKALIKE_SECTION, 1)[1].splitlines()
    start = next(i for i, line in enumerate(lines) if line.startswith(LOOKALIKE_HEADER))
    rows = []
    for line in lines[start + 2:]:
        if not line.startswith("|"):
            break
        rows.append(line.strip("|").split("|")[-1].strip())
    assert len(rows) == len(LOOKALIKE_ROWS), f"the M1 lookalike table has {len(rows)} rows, not {len(LOOKALIKE_ROWS)}"
    features = {}
    for row, (members, cell) in enumerate(zip(LOOKALIKE_ROWS, rows)):
        text = REPORT_CITATION.sub(r"(M1 report ref \1)", GUIDELINE_CITATION.sub(rf"({GUIDELINE}, \1)", cell))
        text = re.sub(r"\*([^*]+)\*", r"\1", text)
        for theirs, ours in LOOKALIKE_REWORDING.get(row, ()):
            assert theirs in text, f"row {row + 1} of the M1 lookalike table no longer reads {theirs!r}"
            text = text.replace(theirs, ours)
        for species in members:
            features.setdefault(species, []).append(([other for other in members if other != species], text))
    return features


def species_cards(classes_csv=CLASSES_CSV, medical_csv=MEDICAL_CSV, name_sources=(M1_REPORT, M2_REPORT),
                  report=M1_REPORT):
    """One card per class, in class_id order, in our own words from repo data only."""
    classes = pd.read_csv(classes_csv).sort_values("class_id")
    medical = pd.read_csv(medical_csv, keep_default_na=False).set_index("species")
    names, features = common_names(name_sources), lookalike_features(report)

    def named(species):
        return f"{species} ({names[species]})" if species in names else species

    rows = []
    for record in classes.itertuples(index=False):
        danger = medical.loc[record.species]
        dangerous = str(danger["dangerous"]) == "True"
        reason = "; ".join(filter(None, (BASIS_NOTE[danger["basis"]], danger["note"])))
        text = (f"{named(record.species)}, family {record.family}. WHO category in India: "
                f"{CATEGORY_TEXT[str(record.who_category)]}. Danger: "
                f"{f'dangerous ({reason})' if dangerous else NOT_LISTED_TEXT}.")
        for others, told_apart in features.get(record.species, []):
            text += f" Told apart from {', '.join(map(named, others))}: {told_apart}."
        rows.append({"id": f"card-{record.class_id:03d}", "class_id": record.class_id, "species": record.species,
                     "common_name": names.get(record.species, ""), "family": record.family,
                     "who_category": str(record.who_category), "dangerous": dangerous, "basis": danger["basis"],
                     "text": text})
    return pd.DataFrame(rows, columns=CARD_COLUMNS)

In [4]:
index, cards = index_passages(PDF), species_cards()
if REBUILD_KB:
    index.to_csv(INDEX_CSV, index=False)
    cards.to_csv(CARDS_CSV, index=False)
same_index = index.to_csv(index=False) == INDEX_CSV.read_text()
same_cards = cards.to_csv(index=False) == CARDS_CSV.read_text()
print(f"index rebuilt from the PDF: {len(index)} passages, the same as {INDEX_CSV.name}: {same_index}")
print(f"cards rebuilt from the repo: {len(cards)} cards, the same as {CARDS_CSV.name}: {same_cards}")
assert same_index and same_cards, "the committed knowledge base is out of date: rerun with REBUILD_KB = True"

index rebuilt from the PDF: 168 passages, the same as who_passages.csv: True
cards rebuilt from the repo: 184 cards, the same as species_cards.csv: True


In [5]:
kb = build_kb(PDF)
index = read_index()
who = [p for p in kb if p.source == WHO_SOURCE]
cards = [p for p in kb if p.source == CARD_SOURCE]
lengths = pd.Series([len(p.text) for p in who])
print(f"{len(who)} guideline passages from {index['page'].nunique()} PDF pages, {len(cards)} species cards")
print("characters per passage: median", int(lengths.median()), "| min", lengths.min(), "| max", lengths.max())
print(index.groupby("topic").agg(passages=("id", "size"), pages=("page", "nunique")).to_string())

168 guideline passages from 45 PDF pages, 184 species cards
characters per passage: median 306 | min 61 | max 892
                passages  pages
topic                          
aftercare             10      4
eye                    9      2
first_aid             28      6
identification         1      1
prevention            22      4
snakes                63     22
symptoms              33      8
transport              2      1


## 2. R2 models: export once, int8

Each model is downloaded at a pinned revision, exported with the TorchScript ONNX exporter (opset 17) and quantised
to dynamic int8 with per-channel weights; per-tensor weights moved embeddings too far from PyTorch (cosine about
0.98 on L3). The int8 model must stay within cosine 0.99 of PyTorch on every check sentence, each encoded alone as
the retriever does. The PyTorch weights and the fp32 ONNX file are deleted after the check, so only the int8 model
stays.

In [6]:
import json
import shutil
from pathlib import Path

import numpy as np

from serpentguard.respond import TOP_K, query_text, text_bytes
from serpentguard.run_outputs import MEBIBYTE, write_json

ONNX_OPSET, ORT_THREADS = 17, 1
FP32_FILE, INT8_FILE, TOKENIZER_FILE, SBERT_CONFIG, EXPORT_JSON = (
    "model_fp32.onnx", "model_int8.onnx", "tokenizer.json", "sentence_bert_config.json", "export.json")
MODEL_INPUTS = ("input_ids", "attention_mask", "token_type_ids")
WEIGHTS_FILE = "model.safetensors"
HF_FILES = ["config.json", WEIGHTS_FILE, TOKENIZER_FILE, "tokenizer_config.json", "special_tokens_map.json",
            "vocab.txt", "modules.json", SBERT_CONFIG, "1_Pooling/config.json"]
CHECK_SENTENCES = ("a snake bit my leg while I was walking home", "how do I keep the arm still",
                   "the photo is too dark to see the head")
EXPORT_MIN_COSINE = 0.99            # int8 against the PyTorch model, per sentence
EMBEDDERS = {"L6": ("sentence-transformers/all-MiniLM-L6-v2", "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"),
             "L3": ("sentence-transformers/paraphrase-MiniLM-L3-v2", "4ca70771034acceecb2e72475f72050fcdde4ddc")}


class EmbeddingRetriever:
    """R2: mean-pooled, L2-normalised sentence embeddings from an int8 ONNX model, ranked by cosine similarity."""

    def __init__(self, kb, model_dir, threads=ORT_THREADS):
        model_dir = Path(model_dir)
        self.passages = list(kb)
        self.files = [model_dir / INT8_FILE, model_dir / TOKENIZER_FILE]
        self.session = _session(self.files[0], threads)
        self.tokenizer = _tokenizer(model_dir)
        self.index = self.encode([p.text for p in self.passages])

    def encode(self, texts):
        return _encode(self.session, self.tokenizer, texts)

    def search(self, query, k=TOP_K):
        scores = self.index @ self.encode([query_text(query)])[0]
        return [self.passages[i] for i in np.argsort(-scores, kind="stable")[:k]]

    def size_bytes(self):
        return text_bytes(self.passages) + sum(path.stat().st_size for path in self.files) + self.index.nbytes


def _tokenizer(model_dir):
    """The model's tokenizer, truncating at max_seq_length and unpadded (some tokenizer files pad every text to 128)."""
    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file(str(Path(model_dir) / TOKENIZER_FILE))
    tokenizer.enable_truncation(json.loads((Path(model_dir) / SBERT_CONFIG).read_text())["max_seq_length"])
    tokenizer.no_padding()
    return tokenizer


def _session(path, threads=ORT_THREADS):
    """An onnxruntime CPU session with telemetry events off: the app runs offline."""
    import onnxruntime as ort
    ort.disable_telemetry_events()
    options = ort.SessionOptions()
    options.intra_op_num_threads, options.inter_op_num_threads = threads, 1
    return ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])


def _encode(session, tokenizer, texts):
    """One text at a time: dynamic int8 scales activations per batch, so batch-mates would change an embedding."""
    names = [node.name for node in session.get_inputs()]
    embeddings = []
    for text in texts:
        arrays = _model_inputs(tokenizer, [text])
        hidden = session.run(None, {name: arrays[name] for name in names})[0]
        embeddings.append(_pool(hidden, arrays["attention_mask"]))
    return np.concatenate(embeddings)


def _model_inputs(tokenizer, texts):
    """Token ids, attention mask and token types for a batch, zero-padded to its longest text."""
    encodings = tokenizer.encode_batch(list(texts))
    width = max(len(e.ids) for e in encodings)
    arrays = {name: np.zeros((len(encodings), width), dtype=np.int64) for name in MODEL_INPUTS}
    for row, encoding in enumerate(encodings):
        for name, values in zip(MODEL_INPUTS, (encoding.ids, encoding.attention_mask, encoding.type_ids)):
            arrays[name][row, :len(values)] = values
    return arrays


def _pool(hidden, mask):
    mask = mask[..., None].astype(np.float32)
    pooled = (hidden * mask).sum(axis=1) / np.clip(mask.sum(axis=1), 1e-9, None)
    return (pooled / np.linalg.norm(pooled, axis=1, keepdims=True)).astype(np.float32)


def model_dir(name, root=Q1_DIR):
    return Path(root) / "models" / name


def export_embedder(name, root=Q1_DIR):
    """The pinned checkpoint exported to ONNX, quantised to dynamic int8 and checked against PyTorch, under root."""
    import torch
    from huggingface_hub import snapshot_download
    from onnxruntime.quantization import QuantType, quantize_dynamic
    from transformers import AutoModel
    repo, revision = EMBEDDERS[name]
    out = model_dir(name, root)
    source = Path(snapshot_download(repo, revision=revision, allow_patterns=HF_FILES, local_dir=out / "hf"))
    model = AutoModel.from_pretrained(source, attn_implementation="eager").eval()

    class LastHidden(torch.nn.Module):
        def __init__(self, inner):
            super().__init__()
            self.inner = inner

        def forward(self, input_ids, attention_mask, token_type_ids):
            return self.inner(input_ids=input_ids, attention_mask=attention_mask,
                              token_type_ids=token_type_ids).last_hidden_state

    module = LastHidden(model)
    for file in (TOKENIZER_FILE, SBERT_CONFIG):
        shutil.copy(source / file, out / file)
    tokenizer = _tokenizer(out)
    arrays = _model_inputs(tokenizer, CHECK_SENTENCES)
    tensors = tuple(torch.from_numpy(arrays[key]) for key in MODEL_INPUTS)
    with torch.no_grad():  # the reference, taken before tracing changes the module
        reference = _pool(module(*tensors).numpy(), arrays["attention_mask"])
    dynamic = {key: {0: "batch", 1: "tokens"} for key in (*MODEL_INPUTS, "last_hidden_state")}
    torch.onnx.export(module, tensors, str(out / FP32_FILE), input_names=list(MODEL_INPUTS),
                      output_names=["last_hidden_state"], dynamic_axes=dynamic, opset_version=ONNX_OPSET, dynamo=False)
    (source / WEIGHTS_FILE).unlink()  # the ONNX file now holds the weights; disk on the shared machine is short
    quantize_dynamic(str(out / FP32_FILE), str(out / INT8_FILE), weight_type=QuantType.QInt8, per_channel=True)
    cosines = {file: float((_encode(_session(out / file), tokenizer, CHECK_SENTENCES) * reference).sum(axis=1).min())
               for file in (FP32_FILE, INT8_FILE)}
    if cosines[INT8_FILE] < EXPORT_MIN_COSINE:
        raise ValueError(f"{name}: the ONNX embeddings drift from PyTorch (min cosine per file: {cosines})")
    report = {"name": name, "repo": repo, "revision": revision, "parameters": sum(p.numel() for p in model.parameters()),
              "fp32_mb": (out / FP32_FILE).stat().st_size / MEBIBYTE, "int8_mb": (out / INT8_FILE).stat().st_size / MEBIBYTE,
              "min_cosine_to_pytorch": cosines,
              "max_seq_length": json.loads((out / SBERT_CONFIG).read_text())["max_seq_length"]}
    (out / FP32_FILE).unlink()  # only the int8 model ships
    write_json(out / EXPORT_JSON, report)
    return report

In [7]:
for name in EMBEDDERS:
    if not (model_dir(name) / INT8_FILE).exists():
        export_embedder(name)
exports = pd.DataFrame([json.loads((model_dir(name) / EXPORT_JSON).read_text()) for name in EMBEDDERS])
exports["int8_cosine_to_pytorch"] = [c[INT8_FILE] for c in exports["min_cosine_to_pytorch"]]
if "int8_cosine_measured" not in exports:
    exports["int8_cosine_measured"] = None
exports["int8_cosine_measured"] = exports["int8_cosine_measured"].fillna("at export, the check sentences in one batch")
print(exports[["name", "repo", "parameters", "max_seq_length", "fp32_mb", "int8_mb", "int8_cosine_to_pytorch"]]
      .to_string(index=False, float_format="%.4f"))
print("threshold", EXPORT_MIN_COSINE)
for row in exports.itertuples():
    print(f"{row.name}: int8 cosine measured {row.int8_cosine_measured}")

name                                          repo  parameters  max_seq_length  fp32_mb  int8_mb  int8_cosine_to_pytorch
  L6        sentence-transformers/all-MiniLM-L6-v2    22713216             256  86.1798  21.8915                  0.9972
  L3 sentence-transformers/paraphrase-MiniLM-L3-v2    17389824             128  65.8292  16.6390                  0.9985
threshold 0.99
L6: int8 cosine measured at export, the check sentences in one batch
L3: int8 cosine measured at export, the check sentences in one batch


## 3. Q1: the four retrievers on the 60 draft questions

Recall@k is the share of questions with at least one gold passage in the top k. MRR uses the first gold passage in
the top 10. There are three question styles: `verdict` questions are asked from a verdict screen, `keyword` ones share
the gold passage's key term, and `paraphrase` ones share no content word with any gold passage.

In [8]:
import platform
import statistics
import subprocess
import time
from importlib import metadata

import pandas as pd

from serpentguard.respond import (BM25_B, BM25_K1, CARD_SOURCE, CARDS_CSV, INDEX_CSV, TOP_K, WHO_SOURCE, FIRST_AID_IDS,
                                  BM25Retriever, LookupRetriever, Question, build_kb, find_who_pdf)
from serpentguard.run_outputs import MEBIBYTE, file_sha256, write_json

RETRIEVAL_CSV, RANKS_CSV, PAIRED_CSV, VERSIONS_JSON = "retrieval.csv", "ranks.csv", "paired.csv", "versions.json"
MRR_CUTOFF, TIMING_REPEATS = 10, 5
LIST_JOIN = ";"   # reasons and gold ids in the questions CSV


def load_questions(path=QUESTIONS_CSV):
    table = pd.read_csv(path, keep_default_na=False)
    return [Question(row.id, row.question, row.style, row.verdict, row.species,
                     tuple(filter(None, row.reason.split(LIST_JOIN))), row.topic, tuple(row.gold_ids.split(LIST_JOIN)))
            for row in table.itertuples(index=False)]


def retrieval_metrics(retriever, questions, k=TOP_K, cutoff=MRR_CUTOFF, repeats=TIMING_REPEATS):
    """Recall@1, Recall@k, MRR@cutoff, median ms per question, MB, each gold rank and the scores by style."""
    retriever.search(questions[0], k=cutoff)
    ranks, times = [], []
    for question in questions:
        found, spent = None, []
        for _ in range(repeats):
            start = time.perf_counter()
            found = retriever.search(question, k=cutoff)
            spent.append(time.perf_counter() - start)
        times.append(statistics.median(spent))
        ranks.append(next((i for i, p in enumerate(found, 1) if p.id in question.gold), None))
    by_style = {style: _scores([r for r, q in zip(ranks, questions) if q.style == style], k)
                for style in sorted({q.style for q in questions})}
    return {**_scores(ranks, k), "ms": 1000 * statistics.fmean(times), "mb": retriever.size_bytes() / MEBIBYTE,
            "by_style": by_style, "ranks": {q.id: rank for q, rank in zip(questions, ranks)}}


def _scores(ranks, k):
    return {"recall1": statistics.fmean(r == 1 for r in ranks),
            "recall3": statistics.fmean(r is not None and r <= k for r in ranks),
            "mrr": statistics.fmean(1 / r if r else 0.0 for r in ranks), "n": len(ranks)}


def paired_sign_tests(ranks, baseline="R1", k=TOP_K):
    """Exact two-sided sign test of each retriever against `baseline` on Recall@k, overall and by style."""
    from scipy.stats import binomtest
    hits = {name: ranks[name].le(k).fillna(False).astype(bool) for name in ranks.columns
            if name not in ("id", "style", "verdict", "topic")}
    rows = []
    for name in hits:
        if name == baseline:
            continue
        for style in ("all", *sorted(ranks["style"].unique())):
            rows_in = ranks["style"].eq(style) if style != "all" else ranks["style"].notna()
            ours, theirs = hits[name][rows_in], hits[baseline][rows_in]
            only_ours, only_theirs = int((ours & ~theirs).sum()), int((~ours & theirs).sum())
            p = binomtest(only_ours, only_ours + only_theirs).pvalue if only_ours + only_theirs else float("nan")
            rows.append({"retriever": name, "baseline": baseline, "style": style, "n": int(rows_in.sum()),
                         "only_retriever": only_ours, "only_baseline": only_theirs, "p_sign_test": p})
    return pd.DataFrame(rows)


def evaluate(pdf=None, questions_csv=QUESTIONS_CSV, out_dir=RESULTS_DIR, q1_dir=Q1_DIR):
    """Q1: R0, R1 and each exported R2 on the questions; writes the four Q1 files and returns the table."""
    pdf = Path(pdf) if pdf else find_who_pdf()
    kb, questions = build_kb(pdf), load_questions(questions_csv)
    retrievers = {"R0": LookupRetriever(kb), "R1": BM25Retriever(kb)}
    for name in EMBEDDERS:
        if (model_dir(name, q1_dir) / INT8_FILE).exists():
            retrievers[f"R2-{name}"] = EmbeddingRetriever(kb, model_dir(name, q1_dir))
    rows, ranks = [], pd.DataFrame({"id": [q.id for q in questions], "style": [q.style for q in questions],
                                    "verdict": [q.verdict for q in questions], "topic": [q.topic for q in questions]})
    for name, retriever in retrievers.items():
        found = retrieval_metrics(retriever, questions)
        ranks[name] = pd.array([found["ranks"][q.id] for q in questions], dtype="Int64")
        for style, scores in [("all", found), *found["by_style"].items()]:
            rows.append({"retriever": name, "style": style, **{key: scores[key] for key in ("recall1", "recall3", "mrr", "n")},
                         "ms": found["ms"], "mb": found["mb"]})
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    table = pd.DataFrame(rows)
    table.to_csv(out_dir / RETRIEVAL_CSV, index=False, float_format="%.4f")
    ranks.to_csv(out_dir / RANKS_CSV, index=False)
    paired_sign_tests(ranks).to_csv(out_dir / PAIRED_CSV, index=False, float_format="%.4f")
    packages = ("numpy", "pandas", "rank-bm25", "onnxruntime", "tokenizers", "transformers", "torch")
    exports = {name: json.loads((model_dir(name, q1_dir) / EXPORT_JSON).read_text())
               for name in EMBEDDERS if (model_dir(name, q1_dir) / EXPORT_JSON).exists()}
    write_json(out_dir / VERSIONS_JSON, {
        "python": platform.python_version(), "machine": platform.machine(), "platform": platform.platform(),
        "packages": {name: metadata.version(name) for name in packages},
        "pdftotext": subprocess.run(["pdftotext", "-v"], capture_output=True, text=True).stderr.splitlines()[0],
        "who_pdf_sha256": file_sha256(pdf), "index_sha256": file_sha256(INDEX_CSV),
        "questions_sha256": file_sha256(questions_csv), "cards_sha256": file_sha256(CARDS_CSV),
        "model_sha256": {name: {file: file_sha256(model_dir(name, q1_dir) / file) for file in (INT8_FILE, TOKENIZER_FILE)}
                         for name in exports},
        "kb": {"who_passages": sum(p.source == WHO_SOURCE for p in kb), "cards": sum(p.source == CARD_SOURCE for p in kb)},
        "config": {"top_k": TOP_K, "mrr_cutoff": MRR_CUTOFF, "timing_repeats": TIMING_REPEATS,
                   "bm25": {"k1": BM25_K1, "b": BM25_B}, "ort_threads": ORT_THREADS, "first_aid_ids": FIRST_AID_IDS,
                   "retrievers": list(retrievers)},
        "embedders": exports})
    return table

In [9]:
table = evaluate(PDF)
print(table.to_string(index=False, float_format="%.3f"))

retriever      style  recall1  recall3   mrr  n    ms     mb
       R0        all    0.100    0.400 0.265 60 0.001  0.039
       R0    keyword    0.050    0.250 0.157 20 0.001  0.039
       R0 paraphrase    0.100    0.350 0.242 20 0.001  0.039
       R0    verdict    0.150    0.600 0.396 20 0.001  0.039
       R1        all    0.417    0.467 0.455 60 0.178  0.214
       R1    keyword    0.900    0.950 0.935 20 0.178  0.214
       R1 paraphrase    0.050    0.050 0.057 20 0.178  0.214
       R1    verdict    0.300    0.400 0.372 20 0.178  0.214
    R2-L6        all    0.400    0.683 0.573 60 1.818 22.940
    R2-L6    keyword    0.500    0.800 0.670 20 1.818 22.940
    R2-L6 paraphrase    0.200    0.600 0.443 20 1.818 22.940
    R2-L6    verdict    0.500    0.650 0.606 20 1.818 22.940
    R2-L3        all    0.333    0.533 0.462 60 0.959 17.687
    R2-L3    keyword    0.450    0.700 0.595 20 0.959 17.687
    R2-L3 paraphrase    0.250    0.450 0.384 20 0.959 17.687
    R2-L3    verdict    

In [10]:
pivot = table.pivot(index="retriever", columns="style", values="recall3").loc[["R0", "R1", "R2-L3", "R2-L6"]]
sizes = table[table["style"] == "all"].set_index("retriever")[["ms", "mb"]]
print("Recall@3 by style, with ms per question and MB on the device")
print(pivot.join(sizes).to_string(float_format="%.3f"))

Recall@3 by style, with ms per question and MB on the device
            all  keyword  paraphrase  verdict    ms     mb
retriever                                                 
R0        0.400    0.250       0.350    0.600 0.001  0.039
R1        0.467    0.950       0.050    0.400 0.178  0.214
R2-L3     0.533    0.700       0.450    0.450 0.959 17.687
R2-L6     0.683    0.800       0.600    0.650 1.818 22.940


### Paired comparison against R1 on Recall@3

Our cost rule: R2 must clearly beat R1 on Recall@3 to pay for its MB in the 30 MB package. Both retrievers
answer the same 60 questions, so the comparison counts the questions only one of them gets right and applies an
exact two-sided sign test to those.

In [11]:
paired = pd.read_csv(RESULTS_DIR / PAIRED_CSV)
print(paired.to_string(index=False, float_format="%.4f"))

retriever baseline      style  n  only_retriever  only_baseline  p_sign_test
       R0       R1        all 60              14             18       0.5966
       R0       R1    keyword 20               0             14       0.0001
       R0       R1 paraphrase 20               7              1       0.0703
       R0       R1    verdict 20               7              3       0.3438
    R2-L6       R1        all 60              20              7       0.0192
    R2-L6       R1    keyword 20               1              4       0.3750
    R2-L6       R1 paraphrase 20              11              0       0.0010
    R2-L6       R1    verdict 20               8              3       0.2266
    R2-L3       R1        all 60              13              9       0.5235
    R2-L3       R1    keyword 20               0              5       0.0625
    R2-L3       R1 paraphrase 20               8              0       0.0078
    R2-L3       R1    verdict 20               5              4       1.0000

In [12]:
ranks = pd.read_csv(RESULTS_DIR / RANKS_CSV)
hit = ranks[["R0", "R1", "R2-L3", "R2-L6"]].le(TOP_K)
missed = ranks[~hit.any(axis=1)]
print(f"{len(missed)} questions no retriever answers in its top {TOP_K}:")
print(missed[["id", "style", "verdict", "topic"]].to_string(index=False))

3 questions no retriever answers in its top 3:
 id      style   verdict          topic
P08 paraphrase dangerous       symptoms
P09 paraphrase   unknown identification
P20 paraphrase   unknown        species


## Reading the result

- R0 needs no model and always shows first aid, but it ignores what was asked; it is the floor.
- R1 is tiny and fast and wins on keyword questions, but it finds almost nothing when the question shares no word
  with the passage, which is how a frightened person often asks.
- R2-L6 is the best on Recall@3 overall and on paraphrases, at about 23 MB with the index and passages. With 4 to
  11 MB budgeted for the encoder, heads and mask head, that leaves no room in a 30 MB package. R2-L3 is about 5 MB
  smaller; it gains over R1 on paraphrases but not overall.
- The species card on a verdict screen comes from `species_card`: for a dangerous verdict, the highest-ranked top-3
  species that is dangerous, or no card, so a card never says "not on the WHO list" under a dangerous verdict.
- The 60 questions and their gold passages are drafts we wrote ourselves, and the report's Q1 numbers come from them.
  The team reviews them, and we rerun Q1 if any of them change.